In [ ]:
!pip install -q langchain-huggingface langchain-chroma langchain-core langchain-text-splitters langchain-community langchain-classic langchain-groq sentence-transformers transformers chromadb panel param


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


# Chat

In [ ]:
import panel as pn
pn.extension()

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from google.colab import drive
import torch

drive.mount("/content/drive")

persist_directory = "/content/drive/MyDrive/law_chatbot_chroma"
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)


/tmp/ipykernel_1045/3075676219.py:2: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

### Memory + ConversationalRetrievalChain

In [ ]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.prompts import PromptTemplate

SYSTEM_TEMPLATE = """انت مساعد قانوني متخصص في القانون البحريني. استخدم المقاطع القانونية التالية فقط للاجابة على السؤال في نهاية النص.

قواعد صارمة يجب اتباعها:
- استند فقط الى النصوص المرفقة، ولا تخترع اي معلومة غير موجودة فيها.
- اذا لم تكن الاجابة موجودة في النصوص المرفقة، صرح بذلك بوضوح ولا تخمن.
- اذكر المصدر الدقيق لكل معلومة (رقم المادة او رقم القضية).

النصوص القانونية:
{context}

السؤال: {question}

الاجابة القانونية المدعومة بالمصادر:"""

QA_CHAIN_PROMPT = PromptTemplate.from_template(SYSTEM_TEMPLATE)

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 2})

qa = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": QA_CHAIN_PROMPT},
)

/tmp/ipykernel_1045/1672934357.py:21: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [ ]:
question = "هل يجوز لصاحب العمل فصل عامل تغيب عن العمل دون انذار؟"
result = qa.invoke({"question": question})
result["answer"]

'**الإجابة:** لا يجوز لصاحب العمل فصل عامل تغيب عن العمل **دون إنذار كتابي مسبق**.  \n\n**المصدران القانونيان الداعمان لهذا الحكم هما:**\n\n1. **المادة 113 (فقرة 4) من قانون العمل رقم 23 لسنة 1976**  \n   - تنص على أن *«يجوز لصاحب العمل فصل العامل دون إخطار أو تعويض إذا تغيب عن العمل بدون سبب مشروع أكثر من عشرين يوماً متقطعة خلال السنة الواحدة أو أكثر من عشرة أيام متوالية، على أن يسبق الفصل إنذار كتابي من صاحب العمل بعد غيابه عشرة أيام في الحالة الأولى وانقطاعه خمسة أيام في الحالة الثانية»* (نص مكرر في نص الحكم المستشهد به).  \n   - إذن، الشرط الأساسي لتبرير الفصل في حالة الغياب هو **وجود إنذار كتابي**؛ بدون هذا الإنذار لا يتحقق أحد الشروط القانونية للفصل دون تعويض.\n\n2. **المادة (107) من قانون العمل**  \n   - تنص على أن أحد الحالات التي يجوز فيها لصاحب العمل إنهاء عقد العمل *«بدون إخطار أو تعويض»* هي *«غياب العامل عن العمل دون سبب مشروع مدة تزيد على عشرين يوماً متقطعة أو عشرة أيام متصلة في السنة الواحدة، على أن يسبق الإنهاء توجيه إنذار كتابي من صاحب العمل بعد غياب العامل عشرة أيام في

In [ ]:
follow_up = "ما هي المدة التي يجب ان يغيبها العامل حتى يجوز فصله؟"
result = qa.invoke({"question": follow_up})
result["answer"]

## Create a chatbot that works on our legal documents

In [ ]:
import param

def load_qa_chain(llm_model="openai/gpt-oss-120b"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
    vectordb = Chroma(persist_directory="/content/drive/MyDrive/law_chatbot_chroma", embedding_function=embedding)
    llm = ChatGroq(model=llm_model, temperature=0)
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    return ConversationalRetrievalChain.from_llm(
        llm,
        retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 2}),
        memory=memory,
        return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": QA_CHAIN_PROMPT},
    )


class cbfs(param.Parameterized):
    chat_history = param.List([])
    answer = param.String("")
    db_response = param.List([])

    def __init__(self, **params):
        super(cbfs, self).__init__(**params)
        self.panels = []
        self.qa = load_qa_chain()

    def convchain(self, query):
        if not query:
            return pn.WidgetBox(pn.Row("المستخدم:", pn.pane.Markdown("", width=600)), scroll=True)
        result = self.qa.invoke({"question": query})
        self.chat_history.extend([(query, result["answer"])])
        self.db_response = result.get("source_documents", [])
        self.answer = result["answer"]
        self.panels.extend([
            pn.Row("المستخدم:", pn.pane.Markdown(query, width=600)),
            pn.Row("المساعد القانوني:", pn.pane.Markdown(self.answer, width=600, styles={"background-color": "#F6F6F6"})),
        ])
        inp.value = ""
        return pn.WidgetBox(*self.panels, scroll=True)

    @param.depends("db_response")
    def get_sources(self):
        if not self.db_response:
            return
        rlist = [pn.Row(pn.pane.Markdown("المصادر المسترجعة:", styles={"background-color": "#F6F6F6"}))]
        for doc in self.db_response:
            rlist.append(pn.Row(pn.pane.Str(f"{doc.metadata} — {doc.page_content[:150]}")))
        return pn.WidgetBox(*rlist, width=600, scroll=True)

    def clr_history(self, count=0):
        self.chat_history = []
        self.panels = []
        return

### Dashboard

In [ ]:
cb = cbfs()

button_clearhistory = pn.widgets.Button(name="مسح المحادثة", button_type="warning")
button_clearhistory.on_click(cb.clr_history)
inp = pn.widgets.TextInput(placeholder="اكتب سؤالك القانوني هنا...")

conversation = pn.bind(cb.convchain, inp)

tab1 = pn.Column(
    pn.Row(inp),
    pn.layout.Divider(),
    pn.panel(conversation, loading_indicator=True, height=400),
)
tab2 = pn.Column(pn.panel(cb.get_sources))
tab3 = pn.Column(
    pn.Row(button_clearhistory, pn.pane.Markdown("يمسح سجل المحادثة لبدء موضوع جديد")),
)

dashboard = pn.Column(
    pn.Row(pn.pane.Markdown("# المساعد القانوني - Capital Legal Base")),
    pn.Tabs(("المحادثة", tab1), ("المصادر", tab2), ("الإعدادات", tab3)),
)
dashboard

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Column
    [0] Row
        [0] Markdown(str)
    [1] Tabs
        [0] Column
            [0] Row
                [0] TextInput(placeholder='اكتب سؤالك القانوني ه...)
            [1] Divider()
            [2] ParamFunction(function, _pane=WidgetBox, defer_load=False, height=400, loading_indicator=True)
        [1] Column
            [0] ParamMethod(method, _pane=Str, defer_load=False)
        [2] Column
            [0] Row
                [0] Button(button_type='warning', color='warning', label='مسح المحادثة', name='مسح المحادثة')
                [1] Markdown(str)

### Note on the demo environment